# M05 lab solution: report reader

**Brief.** Answer five questions about a chart-heavy PDF page by prompting a multimodal model with the page image.

The page is page 1 of the Congressional Research Service report *Wildfire Statistics* (IF10244, updated June 1, 2023), rendered to `../data/if10244_page1.png`. It holds two charts and a table. `../data/questions.jsonl` has five questions whose answers were read off the page by hand. The model is Gemini (`gemini-2.5-flash`, as in notebook `01a`), called through the `google-genai` SDK.

Key: `GOOGLE_API_KEY`.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../../../../src").resolve()))
load_dotenv()

DATA = Path("../data")

import os
import re

import pandas as pd
from google import genai
from google.genai import types

from lab_kit import load_jsonl

In [ ]:
PAGE = DATA / "if10244_page1.png"
questions = load_jsonl(DATA / "questions.jsonl")
image_part = types.Part.from_bytes(data=PAGE.read_bytes(), mime_type="image/png")
print(PAGE.name, PAGE.stat().st_size, "bytes;", len(questions), "questions")

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-2.5-flash"

## 1. The prompt

The text says what the image holds, asks for printed values only, fixes the output shape and gives the model a way out (`NOT_ON_PAGE`) instead of a guess.

In [ ]:
INSTRUCTIONS = """The image is one page of a U.S. Congressional Research Service report on wildfire statistics.
It contains Figure 1 (a line and area chart), Table 1 and Figure 2 (a bar chart with labelled values).
Answer the question using only values printed on the page; read the printed labels, do not estimate bar lengths.
Reply with only the values asked for, separated by "; ", with no units or other words.
If a value is not printed on the page, reply NOT_ON_PAGE.

Question: {question}"""

def ask(question: str) -> str:
    resp = client.models.generate_content(
        model=MODEL,
        contents=[image_part, INSTRUCTIONS.format(question=question)],
        config=types.GenerateContentConfig(temperature=0))
    return (resp.text or "").strip()

## 2. Ask the five questions and check the answers

An answer passes when every expected value appears among the numbers in the reply, compared as numbers (so `59` matches `59.0`). Substring matching would pass a reply that lists every year on the page.

In [ ]:
def numbers(text: str) -> list[float]:
    return [float(x.replace(",", "")) for x in re.findall(r"\d[\d,]*\.?\d*", text)]

def passes(reply: str, expected: list[str]) -> bool:
    got = numbers(reply)
    return all(any(abs(float(e) - g) < 1e-9 for g in got) for e in expected)

rows = []
for q in questions:
    reply = ask(q["question"])
    rows.append({"id": q["id"], "source": q["source"], "question": q["question"],
                 "expected": q["answer"], "model reply": reply, "correct": passes(reply, q["check"])})
    print(q["id"], reply)

## Results

In [ ]:
results = pd.DataFrame(rows)
print(f"{results['correct'].sum()} of {len(results)} correct with {MODEL}")
results[["id", "source", "expected", "model reply", "correct"]]

## What this shows

- Every question has a value printed on the page, so each answer can be graded line by line instead of by reading a summary.
- Figure 2 prints its values on the bars; reading labels is a different task from estimating a bar's length, and the prompt asks for the first.
- A failed check is worth reading in the image: a misread digit, a value from the wrong figure and a guessed value are three different fixes.